In [1]:
!apt-get update -qq
!apt-get install -y ffmpeg -qq
!pip install pydub -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
from google.colab import files

uploaded = files.upload()

Saving ossetian_audio_16k.wav to ossetian_audio_16k.wav


In [11]:
import shutil
import os

# Удаляем старую папку с фрагментами
if os.path.exists("clips"):
    shutil.rmtree("clips")

# Удаляем старую таблицу
if os.path.exists("metadata.csv"):
    os.remove("metadata.csv")

# Удаляем старый архив, если он был
if os.path.exists("ossetian_clips.zip"):
    os.remove("ossetian_clips.zip")

print("Старые файлы удалены.")

Старые файлы удалены.


In [12]:
# Импортируем библиотеку для работы с аудио
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
import csv


# Название исходного аудиофайла, который будем нарезать
INPUT_AUDIO = "ossetian_audio_16k.wav"
OUTPUT_DIR = "clips"
METADATA_FILE = "metadata.csv" #файл с инфой о каждом фрагменте


# Параметры нарезки
MIN_SILENCE_LEN = 70 #минимальная длина паузы.

# Порог тишины.
# Всё, что тише этого значения, программа считает паузой.
# Чем больше значение, тем чувствительнее программа к тишине.
SILENCE_THRESH = -40

# Оставляем немного звука до и после фразы.
# Это нужно, чтобы случайно не обрезать начало или конец слова.
# 200 миллисекунд = 0.1 секунды.
KEEP_SILENCE = 100

# Максимальная длина одного фрагмента.
# Если кусок получился длиннее 12 секунд, мы его дополнительно разделим.
MAX_CHUNK_LEN = 12_000

# Минимальная длина одного фрагмента.
# Если кусок короче 1 секунды, мы его не сохраняем, потому что это может быть шум или обрывок.
MIN_CHUNK_LEN = 1_000


# Функция переводит миллисекунды в секунды
def ms_to_sec(ms):
    return round(ms / 1000, 3)


# Функция делит слишком длинный фрагмент на части
def split_long_segment(start, end, max_len):
    parts = []

    # Начинаем с начала длинного фрагмента
    current = start

    while current < end:
        # Берём конец текущей части.
        # Если до конца осталось меньше max_len, берём настоящий конец.
        part_end = min(current + max_len, end)
        parts.append((current, part_end))

        # Переходим к следующей части
        current = part_end

    return parts


# Создаём папку для фрагментов, если её ещё нет
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Загружаю аудио...")
audio = AudioSegment.from_wav(INPUT_AUDIO)

print(f"Длительность аудио: {round(len(audio) / 1000 / 60, 2)} минут")

print("Ищу речевые фрагменты по паузам...")


# detect_nonsilent ищет участки, где звук не является тишиной
nonsilent_ranges = detect_nonsilent(
    audio,
    min_silence_len=MIN_SILENCE_LEN,
    silence_thresh=SILENCE_THRESH,
    seek_step=10
)

print(f"Найдено речевых фрагментов: {len(nonsilent_ranges)}")


# Здесь будем хранить итоговые фрагменты, которые потом сохраним
final_segments = []


# Перебираем все найденные речевые фрагменты
for start, end in nonsilent_ranges:
    # Немного расширяем фрагмент влево и вправо, чтобы не обрезать начало и конец речи
    start = max(0, start - KEEP_SILENCE)
    end = min(len(audio), end + KEEP_SILENCE)

    # Считаем длительность фрагмента
    duration = end - start

    # Если фрагмент слишком короткий, пропускаем его
    if duration < MIN_CHUNK_LEN:
        continue

    # Если фрагмент слишком длинный, делим его на части
    if duration > MAX_CHUNK_LEN:
        final_segments.extend(split_long_segment(start, end, MAX_CHUNK_LEN))

    # Если фрагмент нормальной длины, просто добавляем его в итоговый список
    else:
        final_segments.append((start, end))


# Показываем, сколько фрагментов получилось после всех проверок
print(f"Итоговых фрагментов: {len(final_segments)}")


# Открываем csv-файл, куда будем записывать информацию о фрагментах
with open(METADATA_FILE, "w", newline="", encoding="utf-8") as csvfile:
    # Создаём объект для записи в csv
    writer = csv.writer(csvfile)

    # Записываем заголовки столбцов
    writer.writerow(["id", "audio_path", "start_sec", "end_sec", "duration_sec"])

    # Перебираем все итоговые фрагменты
    for i, (start, end) in enumerate(final_segments, start=1):
        # Создаём id фрагмента, например clip_000001
        clip_id = f"clip_{i:06d}"

        # Создаём имя файла
        filename = f"{clip_id}.wav"

        # Полный путь, куда сохраним аудиофрагмент
        output_path = os.path.join(OUTPUT_DIR, filename)

        # Вырезаем нужный кусок из большого аудио
        chunk = audio[start:end]

        # Сохраняем этот кусок как отдельный wav-файл
        chunk.export(output_path, format="wav")

        # Записываем информацию о фрагменте в metadata.csv
        writer.writerow([
            clip_id,
            output_path,
            ms_to_sec(start),
            ms_to_sec(end),
            ms_to_sec(end - start)
        ])

print("Готово.")
print(f"Фрагменты сохранены в папку: {OUTPUT_DIR}")
print(f"Метаданные сохранены в файл: {METADATA_FILE}")

Загружаю аудио...
Длительность аудио: 77.11 минут
Ищу речевые фрагменты по паузам...
Найдено речевых фрагментов: 5047
Итоговых фрагментов: 1820
Готово.
Фрагменты сохранены в папку: clips
Метаданные сохранены в файл: metadata.csv


In [13]:
from IPython.display import Audio, display
import os

for filename in sorted(os.listdir("clips"))[:5]:
    print(filename)
    display(Audio(os.path.join("clips", filename)))


clip_000001.wav


clip_000002.wav


clip_000003.wav


clip_000004.wav


clip_000005.wav
